In [1]:
import os
import json

In [2]:
json_data = []

with open('track_ids.json') as json_file:
   json_data = json.load(json_file)

In [3]:
#json_data

In [4]:
import musicbrainzngs
import time

In [5]:
#MusicBrainz API rate-limits -> 1 request per second
'''
For "anonymous" user-agents (see below): we allow through (on average) 50 requests per second, and decline (http 503) the rest.

Source IP address
Unless you have agreed otherwise with MusicBrainz, the rule is as follows:

The rate at which your IP address is making requests is measured. If that rate is too high,
all your requests will be declined (http 503) until the rate drops again. Currently that rate is (on average) 1 request per second.

For example: if your requests are coming in at 4 requests per second, we don't honour 25% of them and decline the other 75% - 
we decline 100% of them, until the rate drops to 1 per second or lower.
'''

'\nFor "anonymous" user-agents (see below): we allow through (on average) 50 requests per second, and decline (http 503) the rest.\n\nSource IP address\nUnless you have agreed otherwise with MusicBrainz, the rule is as follows:\n\nThe rate at which your IP address is making requests is measured. If that rate is too high,\nall your requests will be declined (http 503) until the rate drops again. Currently that rate is (on average) 1 request per second.\n\nFor example: if your requests are coming in at 4 requests per second, we don\'t honour 25% of them and decline the other 75% - \nwe decline 100% of them, until the rate drops to 1 per second or lower.\n'

In [6]:
# 1) Set a distinctive User-Agent
musicbrainzngs.set_useragent(
    "MyAwesomeTagger", 
    "1.0",
    contact="grkritsovas@gmail.com"
)

def find_recording_mbid(track_title, artist_name):
    """
    Searches MusicBrainz by track title + artist name.
    Returns the MBID of the *first* matching recording, or None if none found.
    """
    query = f'recording:"{track_title}" AND artist:"{artist_name}"'
    try:
        result = musicbrainzngs.search_recordings(query=query, limit=1)
    except musicbrainzngs.musicbrainz.NetworkError:
        # handle network issues, rate limiting, etc.
        return None
    
    recordings = result.get("recording-list", [])
    if not recordings:
        return None
    
    # The first recording in the list
    first_rec = recordings[0]
    return first_rec["id"]  # MBID


In [7]:
mbids = []

In [19]:
for i in range(3200, 4400):
    track_title, artist_name = json_data[1], json_data[2][0]
    mbids.append(find_recording_mbid(track_title, artist_name))
    time.sleep(1)

In [21]:
total_found = 0 #have searched the first 4200 songs-found 0 MBIDs
for mbid in mbids:
    if mbid:
        total_found += 1
total_found

0